# Mixed Bin: Offline Visual Search for Apparel Robots

Companion notebook for *"Solving the Mixed Bin: Offline Visual Search for Apparel Robotics."*

It runs end to end on CPU with **no model downloads** using a deterministic `FakeEmbedder`, then shows the one-line swap to the real on-device encoder (Apple **MobileCLIP2**). The retrieval layer is a **Qdrant Edge multivector shard** running inside this process. No server, no network, no client.

Repo: https://github.com/rayl15/qdrant-edge-warehouse-robot

In [ ]:
# On Colab, install straight from GitHub. Locally, `pip install -e '.[dev]'`.
!pip install -q 'mixed-bin @ git+https://github.com/rayl15/qdrant-edge-warehouse-robot.git'

## 1. Build a multivector catalog

Each SKU is stored as **several reference views** (front / folded / crumpled). That is the whole trick: apparel is deformable, so one product shot is brittle.

In [ ]:
import tempfile
import numpy as np
from mixed_bin.embeddings import FakeEmbedder
from mixed_bin.index import EdgeShardIndex, SkuRecord

emb = FakeEmbedder(dim=64)
catalog = [
    ('SKU-1001', 'navy crew tshirt'),
    ('SKU-1002', 'black hoodie'),
    ('SKU-1003', 'red polo'),
    ('SKU-1004', 'white oxford shirt'),
]
records = [
    SkuRecord(sku, title, np.stack([emb.embed_label(sku, view=v) for v in (1, 2, 3)]))
    for sku, title in catalog
]

tmp = tempfile.mkdtemp()
index = EdgeShardIndex(f'{tmp}/shard', dim=64)
index.build(records)
print(f'Indexed {len(records)} SKUs as multivector points.')

## 2. Plan picks for a mixed bin

The robot detects each garment, embeds the crop, and searches its local Edge shard. Because retrieval runs in-process, it is the cheapest step in the loop.

In [ ]:
import time
from PIL import Image
from mixed_bin.config import Settings
from mixed_bin.detector import Crop
from mixed_bin.search import MixedBinPicker

class LabelledDetector:
    def __init__(self, items):
        self.items = items
    def detect(self, _bin):
        crops = []
        for label, view, box in self.items:
            img = Image.new('RGB', (64, 64)); img.info.update(label=label, view=view)
            crops.append(Crop(image=img, box=box))
        return crops

# Three garments, each at an unfamiliar angle (views 7/8/9 were never indexed).
bin_items = [
    ('SKU-1003', 7, (20, 15, 180, 175)),
    ('SKU-1001', 8, (200, 40, 360, 200)),
    ('SKU-1004', 9, (60, 210, 220, 370)),
]
picker = MixedBinPicker(LabelledDetector(bin_items), emb, index, Settings(min_confidence=0.3))

start = time.perf_counter()
targets = picker.plan_picks(Image.new('RGB', (400, 400)))
elapsed = (time.perf_counter() - start) * 1000
for t in targets:
    print(f'{t.sku:10} {t.title:20} score={t.confidence:.3f} grip={t.grip_point}')
print(f'\nParsed + searched {len(targets)} garments in {elapsed:.1f} ms (in-process).')

## 3. Why multivector matters

Store only the *front* view and a crumpled crop can lose to an unrelated SKU. Add the crumpled view and MAX_SIM recovers the right match. This is `test_multivector_beats_single_view_for_deformed_item` in the repo, inline.

In [ ]:
def unit(v):
    return (v / np.linalg.norm(v)).astype('float32')

rng = np.random.default_rng(0)
base_a = unit(rng.standard_normal(64))
deform = unit(rng.standard_normal(64))
front_a, crumpled_a = base_a, unit(0.5 * base_a + 0.9 * deform)
query = unit(0.35 * base_a + 1.0 * deform)
e = rng.standard_normal(64); e = unit(e - (e @ query) * query)
front_b = unit(0.6 * query + np.sqrt(1 - 0.36) * e)

single = EdgeShardIndex(f'{tmp}/single', dim=64)
single.build([SkuRecord('SKU-A', 'a', np.stack([front_a])), SkuRecord('SKU-B', 'b', np.stack([front_b]))])
multi = EdgeShardIndex(f'{tmp}/multi', dim=64)
multi.build([SkuRecord('SKU-A', 'a', np.stack([front_a, crumpled_a])), SkuRecord('SKU-B', 'b', np.stack([front_b]))])

print('front-only  ->', single.search(query, top_k=1)[0].sku, '(wrong: distractor wins)')
print('multivector ->', multi.search(query, top_k=1)[0].sku, '(right: crumpled view rescues it)')

## 4. The real encoder (optional)

Everything above used `FakeEmbedder`. Swap in MobileCLIP2 and the pipeline code does not change. On a Jetson-class board this runs fully offline once weights are cached.

```python
# pip install open_clip_torch torch
from mixed_bin.embeddings import MobileClipEmbedder
emb = MobileClipEmbedder('hf-hub:timm/MobileCLIP2-S0-OpenCLIP')  # 512-d
```

Then build the catalog from a folder of product photos with `mixed_bin.catalog.build_records`, and detect real garments with `YoloBinDetector`.